# Fair Benchmark — Clean Raw-Text BPC (no format artefact)

Önceki benchmark'lar **haksızdı**:
- `manual_ppl` packed chunk → bizim training format'ımıza tam uyumlu, vanilla'ya haksız
- `compute_bpc` decode→repack → bizim modele haksız (EOS strip + BOS artefakt)
- val set'te %0.34 garbled vardı (temizlendi)

**Bu test adil:** 2000 ham clean Türkçe akademik abstract (`{title}\n\n{abstract}`). Her model **kendi tokenizer'ı** ile naturel tokenize eder. BPC (bits-per-char) tokenizer-bağımsız → cross-model adil.

**Rakipler:** Vanilla Trendyol-7b-base, Trendyol-7b-chat, Cosmos-Turkish-8B, trakad-7b-base (ours).

**Runtime:** A100 önerilir (~40 dk), T4 da olur (~70 dk). Training yok, sadece forward pass.

## 1. Install (BİR KEZ → RESTART)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!nvidia-smi
!pip install -q "transformers==4.45.2" "tokenizers>=0.20,<0.21" sentencepiece pyarrow pandas
print('\n*** RESTART SESSION NOW ***')

## 2. (Restart sonrası) Env + clean eval set indir

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import pandas as pd, urllib.request

URL = 'https://raw.githubusercontent.com/hakansabunis/tr-academic-research-agent/main/data/derived/clean_eval.parquet'
urllib.request.urlretrieve(URL, '/content/clean_eval.parquet')
eval_df = pd.read_parquet('/content/clean_eval.parquet')
texts = eval_df['text'].tolist()
print(f'Clean eval texts: {len(texts):,}')
print(f'Avg chars: {sum(len(t) for t in texts)/len(texts):.0f}')
print(f'Source mix: {eval_df["source"].value_counts().to_dict()}')
print(f'\nSample (first 300 chars):\n{texts[0][:300]}')

## 3. BPC fonksiyonu (her model kendi tokenizer'ı ile)

In [ ]:
import torch, math, time, gc

def compute_bpc(model, tok, texts, max_seq=2048, log_every=400):
    """Bits-per-character — tokenizer-independent. Natural tokenization,
    no packing, no decode-repack. Fair across models."""
    model.eval()
    total_nll = 0.0  # natural log nats
    total_chars = 0
    t0 = time.time()
    with torch.no_grad():
        for i, text in enumerate(texts):
            enc = tok(text, return_tensors='pt', truncation=True, max_length=max_seq)
            ids = enc['input_ids'].to(model.device)
            if ids.shape[1] < 2:
                continue
            out = model(input_ids=ids, labels=ids)
            n_tok = ids.shape[1]
            total_nll += out.loss.item() * (n_tok - 1)
            total_chars += len(text)
            if (i + 1) % log_every == 0:
                bpc = total_nll / total_chars / math.log(2)
                eta = (len(texts) - i - 1) * (time.time() - t0) / (i + 1)
                print(f'  [{i+1}/{len(texts)}] BPC={bpc:.4f} ETA {eta/60:.1f}m')
    return total_nll / total_chars / math.log(2)

RESULTS = {}

## 4. Vanilla Trendyol-7b-base

In [ ]:
from transformers import AutoModelForCausalLM, LlamaTokenizer

tok_tr = LlamaTokenizer.from_pretrained('Trendyol/Trendyol-LLM-7b-base-v1.0')
if tok_tr.pad_token is None:
    tok_tr.pad_token = tok_tr.eos_token

m = AutoModelForCausalLM.from_pretrained(
    'Trendyol/Trendyol-LLM-7b-base-v1.0', torch_dtype=torch.bfloat16).cuda()
RESULTS['vanilla_trendyol_7b'] = compute_bpc(m, tok_tr, texts)
print(f"\nVanilla Trendyol BPC: {RESULTS['vanilla_trendyol_7b']:.4f}")
del m; torch.cuda.empty_cache(); gc.collect()

## 5. Trendyol-7b-chat

In [ ]:
m = AutoModelForCausalLM.from_pretrained(
    'Trendyol/Trendyol-LLM-7b-chat-v1.0', torch_dtype=torch.bfloat16).cuda()
RESULTS['trendyol_7b_chat'] = compute_bpc(m, tok_tr, texts)
print(f"\nTrendyol-Chat BPC: {RESULTS['trendyol_7b_chat']:.4f}")
del m; torch.cuda.empty_cache(); gc.collect()

## 6. Cosmos-Turkish-Llama-8B (kendi tokenizer'ı)

In [ ]:
from transformers import AutoTokenizer
tok_cos = AutoTokenizer.from_pretrained('ytu-ce-cosmos/Turkish-Llama-8b-DPO-v0.1')
m = AutoModelForCausalLM.from_pretrained(
    'ytu-ce-cosmos/Turkish-Llama-8b-DPO-v0.1', torch_dtype=torch.bfloat16).cuda()
RESULTS['cosmos_turkish_8b'] = compute_bpc(m, tok_cos, texts)
print(f"\nCosmos-Turkish-8B BPC: {RESULTS['cosmos_turkish_8b']:.4f}")
del m; torch.cuda.empty_cache(); gc.collect()

## 7. trakad-7b-base (ours)

Repo config.json bozuk (push artefaktı) — Mistral config explicit + son checkpoint subfolder.

In [ ]:
from transformers import MistralForCausalLM, MistralConfig
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files('hakansabunis/trakad-7b-base')
ckpts = sorted({f.split('/')[0] for f in files if f.startswith('checkpoint-')},
               key=lambda x: int(x.split('-')[1]))
LATEST = ckpts[-1]
print(f'Checkpoints: {ckpts} -> using {LATEST}')

cfg = MistralConfig.from_pretrained('Trendyol/Trendyol-LLM-7b-base-v1.0')
m = MistralForCausalLM.from_pretrained(
    'hakansabunis/trakad-7b-base', subfolder=LATEST, config=cfg,
    torch_dtype=torch.bfloat16).cuda()
# Bizim model Trendyol tokenizer kullanıyor (continued pre-train, vocab değişmedi)
RESULTS['trakad_7b_base_ours'] = compute_bpc(m, tok_tr, texts)
print(f"\ntrakad-7b-base BPC: {RESULTS['trakad_7b_base_ours']:.4f}")
del m; torch.cuda.empty_cache(); gc.collect()

## 8. Adil karşılaştırma tablosu + JSON

In [ ]:
import json

order = [
    ('Vanilla Trendyol-7b-base', 'vanilla_trendyol_7b', '7B'),
    ('Trendyol-7b-chat-v1.0',    'trendyol_7b_chat',    '7B'),
    ('Cosmos-Turkish-Llama-8B',  'cosmos_turkish_8b',   '8B'),
    ('trakad-7b-base (ours)',    'trakad_7b_base_ours', '7B'),
]
print('='*68)
print('FAIR BENCHMARK — Clean raw-text BPC (lower = better, own tokenizers)')
print('='*68)
print(f'{"Model":35s} {"Params":>7s} {"BPC":>9s}')
print('-'*68)
best = min(RESULTS.values())
for name, key, params in order:
    bpc = RESULTS[key]
    mark = '  <- best' if bpc == best else ''
    print(f'{name:35s} {params:>7s} {bpc:>9.4f}{mark}')
print('='*68)

ours = RESULTS['trakad_7b_base_ours']
van = RESULTS['vanilla_trendyol_7b']
print(f'\nOurs vs vanilla Trendyol: {(van-ours)/van*100:+.1f}% '
      f'({"better" if ours<van else "worse"})')

out = {
    'eval_set': 'clean_eval.parquet (2000 raw clean Turkish academic abstracts)',
    'method': 'BPC, each model own tokenizer, natural tokenization (no packing/decode artefact)',
    'results': RESULTS,
}
with open('/content/fair_benchmark.json', 'w') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print('\nSaved /content/fair_benchmark.json — bu sonuçları paylaş.')